In [0]:
# =============================================================================
# NOTEBOOK  : maintenance
# PURPOSE   : Optimize, vacuum, and clean all Delta tables
# LAYER     : Maintenance (runs after daily gold pipeline completes)
# FREQUENCY : Daily (nightly — after all gold notebooks complete)
# PATTERN   : spark.sql OPTIMIZE + VACUUM per table
#             Audit logged to pipeline_audit
#
# OPERATIONS:
#   OPTIMIZE  → compacts small files into larger ones
#               improves query performance significantly
#               safe to run anytime — no data loss
#
#   VACUUM    → removes old Delta log files and deleted data files
#               files older than retention_hours are permanently deleted
#               default retention: 168 hours (7 days)
#               DO NOT run with retention < 168h unless you are sure
#               no active streams or time travel queries need older files
#
#   ZORDER    → co-locates related data in same files
#               run inside OPTIMIZE — speeds up filtered queries
#               applied per table based on most common filter columns
#               only beneficial on tables > 1GB
#
# NOTE: OPTIMIZE and VACUUM are idempotent — safe to rerun
#       VACUUM is irreversible — deleted files cannot be recovered
#       retention_hours set in config.json under pipeline.vacuum_retain_hours
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
PIPELINE         = "maintenance"
AUDIT_TABLE      = cfg["tables"]["pipeline_audit"]
VACUUM_HOURS     = cfg["pipeline"].get("vacuum_retain_hours", 168)
 
# Disable vacuum safety check — we manage retention explicitly via config
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [0]:
# ── Define Tables + ZORDER Columns ────────────────────────────────────
 
# Format: (table_config_key, [zorder_columns] or None)
# ZORDER only applied where queries commonly filter on those columns
# No ZORDER on dim tables — too small to benefit
# No ZORDER on tables < ~1GB — overhead exceeds benefit
 
BRONZE_TABLES = [
    ("bronze_pos_transactions",    ["store_id", "product_id"]),
    ("bronze_warehouse_inventory", ["product_id"]),
    ("bronze_store_inventory",     ["store_id", "product_id"]),
    ("bronze_purchase_orders",     ["supplier_id", "product_id"]),
    ("bronze_customers",           None),
    ("bronze_product_master",      None),
    ("bronze_store_master",        None),
    ("bronze_supplier_master",     None),
]
 
SILVER_TABLES = [
    ("silver_pos_transactions",    ["store_id", "product_id"]),
    ("silver_warehouse_inventory", ["product_id"]),
    ("silver_store_inventory",     ["store_id", "product_id"]),
    ("silver_purchase_orders",     ["supplier_id", "product_id"]),
    ("silver_customers",           ["loyalty_tier"]),
    ("silver_product_master",      ["category", "supplier_id"]),
    ("silver_store_master",        None),
    ("silver_supplier_master",     None),
]
 
GOLD_TABLES = [
    ("gold_fact_inventory_full",      ["store_id", "product_id"]),
    ("gold_fact_inventory_kpis",      ["store_id", "product_id"]),
    ("gold_fact_demand_trends",       ["store_id", "product_id"]),
    ("gold_fact_daily_inventory",     ["store_id", "product_id"]),
    ("gold_fact_customer_sales",      ["store_id", "customer_id"]),
    ("gold_fact_supplier_performance", None),
    ("gold_dim_product",              None),
    ("gold_dim_store",                None),
    ("gold_dim_supplier",             None),
    ("gold_dim_customer",             None),
]
 
PLATFORM_TABLES = [
    ("pipeline_audit", None),
]
 
ALL_TABLES = BRONZE_TABLES + SILVER_TABLES + GOLD_TABLES + PLATFORM_TABLES

In [0]:
# ── Helper Functions ──────────────────────────────────────────────────
 
def optimize_table(table_name, zorder_cols=None):
    """
    Runs OPTIMIZE on a Delta table.
    If zorder_cols provided, co-locates data by those columns.
    Returns metrics dict from Delta history.
    """
    try:
        if zorder_cols:
            zorder_str = ", ".join(zorder_cols)
            spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zorder_str})")
            print(f"  [OPTIMIZE+ZORDER] {table_name} → {zorder_str}")
        else:
            spark.sql(f"OPTIMIZE {table_name}")
            print(f"  [OPTIMIZE] {table_name}")
 
        metrics = (
            spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        files_added   = int(metrics.get("numAddedFiles",   0))
        files_removed = int(metrics.get("numRemovedFiles", 0))
        print(f"    files_added={files_added} | files_removed={files_removed}")
        return {"status": "SUCCESS", "files_added": files_added,
                "files_removed": files_removed}
 
    except Exception as e:
        print(f"  [ERROR] OPTIMIZE failed for {table_name}: {e}")
        return {"status": "FAILED", "error": str(e)}
 
 
def vacuum_table(table_name, retain_hours):
    """
    Runs VACUUM on a Delta table.
    Permanently deletes files older than retain_hours.
    """
    try:
        spark.sql(f"VACUUM {table_name} RETAIN {retain_hours} HOURS")
        print(f"  [VACUUM] {table_name} → retain {retain_hours}h")
        return {"status": "SUCCESS"}
    except Exception as e:
        print(f"  [ERROR] VACUUM failed for {table_name}: {e}")
        return {"status": "FAILED", "error": str(e)}

In [0]:
# ── Run Maintenance ───────────────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE,
                     target_table="all_tables",
                     source_table=None)
 
try:
    optimize_results = {}
    vacuum_results   = {}
    errors           = []
 
    total_tables     = len(ALL_TABLES)
    tables_optimized = 0
    tables_vacuumed  = 0
    total_files_removed = 0
 
    print("=" * 60)
    print(f"MAINTENANCE START — {total_tables} tables")
    print(f"Vacuum retention: {VACUUM_HOURS} hours")
    print("=" * 60)
 
    for config_key, zorder_cols in ALL_TABLES:
        table_name = cfg["tables"].get(config_key)
 
        if not table_name:
            print(f"  [SKIP] {config_key} not found in config — skipping")
            continue
 
        print(f"\n[TABLE] {table_name}")
 
        # OPTIMIZE
        opt_result = optimize_table(table_name, zorder_cols)
        optimize_results[config_key] = opt_result
        if opt_result["status"] == "SUCCESS":
            tables_optimized += 1
            total_files_removed += opt_result.get("files_removed", 0)
        else:
            errors.append(f"OPTIMIZE:{config_key}")
 
        # VACUUM
        vac_result = vacuum_table(table_name, VACUUM_HOURS)
        vacuum_results[config_key] = vac_result
        if vac_result["status"] == "SUCCESS":
            tables_vacuumed += 1
        else:
            errors.append(f"VACUUM:{config_key}")
 
    print("\n" + "=" * 60)
    print(f"MAINTENANCE COMPLETE")
    print(f"  tables_optimized : {tables_optimized}/{total_tables}")
    print(f"  tables_vacuumed  : {tables_vacuumed}/{total_tables}")
    print(f"  files_removed    : {total_files_removed}")
    print(f"  errors           : {len(errors)}")
    if errors:
        print(f"  failed           : {', '.join(errors)}")
    print("=" * 60)
 
    final_status = "SUCCESS" if not errors else "PARTIAL"
 
    end_audit(
        spark, PROJECT_ROOT, run_id, final_status,
        extra={
            "tables_optimized":   str(tables_optimized),
            "tables_vacuumed":    str(tables_vacuumed),
            "total_files_removed": str(total_files_removed),
            "errors":             str(len(errors)),
            "failed_tables":      ", ".join(errors) if errors else "none",
            "vacuum_retain_hours": str(VACUUM_HOURS)
        }
    )
 
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table(AUDIT_TABLE) \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "start_time", "end_time", "extra_metadata") \
    .show(truncate=False)